# Đếm người và khoanh vùng đám đông: MobileCount và MobileCount + Mask, 5-fold cross-validation (ShanghaiTech Part B)

Notebook cho báo cáo: so sánh **MobileCount (baseline)** và **MobileCount + Mask** (thêm nhánh mask đám đông) bằng **cross-validation 5 fold**, cùng dữ liệu, cùng quy trình train và cùng cách đánh giá.

**Quy trình:**
- 400 ảnh train được chia thành **5 fold** (80 ảnh mỗi fold), phân tầng theo số người để fold nào cũng có đủ ảnh thưa và ảnh đông.
- Với mỗi fold k: train trên 4 fold còn lại (320 ảnh), mỗi `VAL_EVERY` epoch đo MAE trên fold k (80 ảnh) và giữ **checkpoint có val MAE thấp nhất**. Sau đó đánh giá checkpoint này trên 316 ảnh test **đúng một lần**.
- Hai model dùng **cùng fold và cùng seed** ở mỗi fold, nên có thể so sánh từng cặp.
- Kết quả báo cáo là **trung bình ± độ lệch chuẩn qua 5 fold**, kèm số fold mà Mask tốt hơn baseline.

Tập test không tham gia vào việc chọn checkpoint. Cách làm này theo khuyến nghị trong README của repo chính thức MobileCount: tách khoảng 10–20% tập train làm validation thay vì chọn trên tập test.

**Chạy trên Kaggle:** Add Input dataset `tthien/shanghaitech`, bật **GPU**, rồi **Save Version → Save & Run All**. Mỗi lần train khoảng 40–45 phút trên T4; đủ 5 fold × 2 model khoảng 7–7,5 giờ. Nếu một phiên không đủ thời gian: notebook tự dừng trước khi vượt `SESSION_HOURS`; tạo version mới, **Add Input** output của version trước, notebook sẽ dùng lại các fold đã xong và chỉ train phần còn thiếu.

**Đầu ra** ở `/kaggle/working`: kết quả từng fold (`results_mc_{model}_f{k}_partB.json`, `preds_…csv`, `history_…csv`), bảng `cv_summary_partB.csv`, `cv_folds_partB.csv`, hình `cv_*.png`, và trọng số cho demo `mc_baseline_partB.pth`, `mc_mask_partB.pth` (fold có **val** MAE thấp nhất của mỗi model).

**Nội dung:** 0. Cấu hình → 1. Dữ liệu (EDA) và chia fold → 2. Density map → 3. Dataset → 4. Kiến trúc → 5. Công cụ train và đánh giá → **6. Train 5 fold × 2 model** → 7. Hình kết quả → 8. Bảng so sánh.

In [ ]:
import contextlib, glob, json, math, os, random, re, shutil, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import display
from PIL import Image
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
if device.type == 'cuda':
    prop = torch.cuda.get_device_properties(0)
    GPU_NAME = prop.name
    print(f'GPU: {prop.name}, {prop.total_memory / 1024**3:.1f} GB')
else:
    GPU_NAME = 'CPU'
    print('CẢNH BÁO: không có GPU. Vào Settings → Accelerator → chọn GPU.')

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## 0. Cấu hình

In [ ]:
# ---- Dữ liệu ----
PART = 'B'                 # ShanghaiTech Part B: ảnh đường phố, mật độ vừa
DATASET_DIR = '/kaggle/input/datasets/tthien/shanghaitech'
INPUT_DIR, WORK_DIR = '/kaggle/input', '/kaggle/working'
FIXED_SIGMA = 15           # độ rộng Gaussian (pixel) khi tạo density map ground truth
DENSITY_SCALE = 100.0      # nhân density cho dễ học; số người = tổng density / DENSITY_SCALE
STRIDE = 4                 # density map bằng 1/4 kích thước ảnh
MASK_THRESH = 1e-6         # ô có density GT > ngưỡng = "có người" (nhãn của nhánh mask)

# ---- Cross-validation ----
K_FOLDS = 5                # 400 ảnh train → 5 fold × 80 ảnh
RUN_FOLDS = list(range(K_FOLDS))   # có thể chia phiên, ví dụ [0, 1, 2] rồi [3, 4]
VAL_EVERY = 5              # đo val MAE mỗi 5 epoch để chọn checkpoint
SESSION_HOURS = 11.0       # không bắt đầu lần train mới nếu có thể vượt mốc này (Kaggle giới hạn 12 giờ)

# ---- Train (giống nhau cho cả hai model) ----
SEED = 42                  # fold k dùng seed SEED + k, giống nhau cho hai model
EPOCHS = 400
LR, LR_MIN = 1e-4, 1e-6    # learning rate giảm theo cosine từ LR xuống LR_MIN
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 8
CROP = 512                 # cạnh patch cắt ngẫu nhiên từ ảnh train
LOG_EVERY = 25             # in log mỗi 25 epoch
MASK_LAMBDA = 0.1          # Mask: trọng số loss của nhánh mask
MAX_HOURS_PER_RUN = 1.5    # giới hạn an toàn cho mỗi model
NUM_WORKERS = 4
QUICK = False              # True: chạy thử vài phút (8 ảnh, 2 epoch) để kiểm tra lỗi

# ---- Hai model ----
VARIANTS = {
    'baseline': dict(label='MobileCount (baseline)', mask=False),
    'mask': dict(label='MobileCount + Mask', mask=True),
}

# Giá trị kỹ thuật cố định (không cần chỉnh)
N_SAMPLES, FPS_SIZE, DATA_LIMIT = 6, (768, 1024), None


def find_data_root(*bases):
    for base in bases:
        for dirpath, dirnames, _ in os.walk(base):
            if 'part_A' in dirnames or 'part_B' in dirnames:
                return dirpath
    raise FileNotFoundError('Không thấy ShanghaiTech: kiểm tra Add Input.')


DATA_ROOT = find_data_root(DATASET_DIR, INPUT_DIR)

if QUICK:
    DATA_LIMIT, EPOCHS, VAL_EVERY, LOG_EVERY, N_SAMPLES, CROP, MAX_HOURS_PER_RUN = 10, 2, 1, 1, 2, 256, 0.1
    RUN_FOLDS = [0, 1]

os.makedirs(WORK_DIR, exist_ok=True)
assert os.path.isdir(os.path.join(DATA_ROOT, f'part_{PART}')), f'Không có part_{PART} trong {DATA_ROOT}'
assert CROP % 32 == 0


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


seed_everything(SEED)
display(pd.DataFrame(VARIANTS).T)
SESSION_START = time.time()
print(f'Part {PART} | {K_FOLDS}-fold CV, chạy fold {RUN_FOLDS} | {EPOCHS} epoch, lr cosine {LR} → {LR_MIN} | '
      f'patch {CROP} | batch {BATCH_SIZE} | checkpoint theo val MAE của fold' + (' | CHẾ ĐỘ CHẠY THỬ' if QUICK else ''))

## 1. Dữ liệu ShanghaiTech

Mỗi ảnh có một file `GT_IMG_x.mat` chứa tọa độ `(x, y)` của từng đầu người. 400 ảnh train được chia thành 5 fold cho cross-validation (mục 1.2); tập test (316 ảnh) chỉ dùng một lần ở cuối mỗi lần train.

In [ ]:
def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', os.path.basename(path))]


def gt_path(img_path):
    split_dir = os.path.dirname(os.path.dirname(img_path))
    name = 'GT_' + os.path.splitext(os.path.basename(img_path))[0] + '.mat'
    for folder in ('ground-truth', 'ground_truth', 'gt'):
        path = os.path.join(split_dir, folder, name)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'Không thấy nhãn cho {img_path}')


def load_points(img_path):
    info = sio.loadmat(gt_path(img_path))['image_info']
    return np.asarray(info[0, 0][0, 0][0], dtype=np.float32).reshape(-1, 2)


def load_split(split):
    pattern = os.path.join(DATA_ROOT, f'part_{PART}', split, 'images', '*.jpg')
    recs = []
    for path in sorted(glob.glob(pattern), key=natural_key)[:DATA_LIMIT]:
        w, h = Image.open(path).size
        recs.append({'name': os.path.basename(path), 'path': path, 'points': load_points(path), 'h': h, 'w': w})
    return recs


def describe(recs):
    c = np.array([len(r['points']) for r in recs])
    return {'số ảnh': len(recs), 'tổng số người': int(c.sum()), 'ít nhất': int(c.min()), 'nhiều nhất': int(c.max()),
            'trung bình': round(c.mean(), 1), 'trung vị': float(np.median(c)), 'độ lệch chuẩn': round(c.std(), 1)}


t0 = time.time()
train_recs, test_recs = load_split('train_data'), load_split('test_data')
test_by_name = {r['name']: r for r in test_recs}
SPLITS = {'train': train_recs, 'test': test_recs}
print(f'Đọc {len(train_recs) + len(test_recs)} ảnh và nhãn trong {time.time() - t0:.1f}s')
display(pd.DataFrame({name: describe(recs) for name, recs in SPLITS.items()}).T)
sizes = pd.Series([f"{r['w']}×{r['h']}" for r in train_recs + test_recs]).value_counts()
print(f'Số kích thước ảnh khác nhau: {len(sizes)}; phổ biến nhất: {sizes.index[0]} ({sizes.iloc[0]} ảnh)')

In [ ]:
counts = {name: np.array([len(r['points']) for r in recs]) for name, recs in SPLITS.items()}
fig, axes = plt.subplots(1, 2, figsize=(16, 4.2))
bins = np.linspace(0, max(c.max() for c in counts.values()), 30)
for name, c in counts.items():
    axes[0].hist(c, bins=bins, alpha=0.55, label=f'{name} ({len(c)} ảnh)')
axes[0].set(title=f'Phân bố số người mỗi ảnh (Part {PART})', xlabel='số người', ylabel='số ảnh')
axes[0].legend()
axes[1].boxplot(list(counts.values()))
axes[1].set_xticks(range(1, len(counts) + 1), list(counts.keys()))
axes[1].set(title='Số người mỗi ảnh theo tập', ylabel='số người')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'data_stats_part{PART}.png'), bbox_inches='tight'); plt.show()

picks = random.Random(0).sample(train_recs, min(4, len(train_recs)))
fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4.2), squeeze=False)
for ax, r in zip(axes[0], picks):
    ax.imshow(Image.open(r['path']).convert('RGB'))
    ax.scatter(r['points'][:, 0], r['points'][:, 1], s=4, c='red')
    ax.set_title(f"{r['name']}: {len(r['points'])} người"); ax.axis('off')
plt.suptitle('Ảnh mẫu với nhãn điểm đầu người (chấm đỏ)'); plt.tight_layout(); plt.show()

### 1.1. Phân tích thêm

- **Vị trí đầu người trên ảnh:** người tập trung ở vùng nào của khung hình (liên quan tới việc chọn vùng giám sát khi triển khai).
- **Khoảng cách tới đầu người gần nhất:** cho biết mức chen chúc, và so với độ rộng Gaussian (`FIXED_SIGMA`) dùng khi tạo density map.
- **Số ảnh theo nhóm mật độ** ở từng tập: các nhóm này được dùng khi phân tích sai số ở phần kết quả.

In [ ]:
BINS = [0, 50, 100, 200, np.inf]
BIN_LABELS = [f'{int(a) + 1 if a else 0}–{int(b)}' if np.isfinite(b) else f'>{int(a)}' for a, b in zip(BINS[:-1], BINS[1:])]


def density_bin(gt):
    return pd.cut(gt, BINS, labels=BIN_LABELS, include_lowest=True)


pos = np.concatenate([r['points'] / [r['w'], r['h']] for r in train_recs if len(r['points'])])
nn_dist = np.concatenate([cKDTree(r['points']).query(r['points'], k=2)[0][:, 1]
                          for r in train_recs if len(r['points']) > 1])

fig, axes = plt.subplots(1, 3, figsize=(22, 4.6))
_, _, _, im = axes[0].hist2d(pos[:, 0], pos[:, 1], bins=[32, 24], range=[[0, 1], [0, 1]], cmap='magma')
axes[0].invert_yaxis(); axes[0].grid(False)
fig.colorbar(im, ax=axes[0], label='số đầu người')
axes[0].set(title='Vị trí đầu người trên ảnh (train)', xlabel='x / chiều rộng', ylabel='y / chiều cao')

axes[1].hist(np.clip(nn_dist, 0, 150), bins=60)
axes[1].axvline(FIXED_SIGMA, color='r', ls='--', label=f'sigma = {FIXED_SIGMA} px')
axes[1].axvline(3 * FIXED_SIGMA, color='orange', ls='--', label=f'3 sigma = {3 * FIXED_SIGMA} px')
axes[1].axvline(np.median(nn_dist), color='k', ls=':', label=f'trung vị = {np.median(nn_dist):.0f} px')
axes[1].set(title='Khoảng cách tới đầu người gần nhất (train, cắt ở 150 px)', xlabel='pixel', ylabel='số đầu người')
axes[1].legend(fontsize=8)

groups = pd.DataFrame({name: density_bin(pd.Series(c)).value_counts().reindex(BIN_LABELS) for name, c in counts.items()})
groups.plot.bar(ax=axes[2], rot=0)
axes[2].set(title='Số ảnh theo nhóm mật độ', xlabel='số người trong ảnh', ylabel='số ảnh')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'eda_part{PART}.png'), bbox_inches='tight'); plt.show()
display(groups.rename_axis('số người').T)

### 1.2. Chia 5 fold

Chia **phân tầng theo số người**: sắp 400 ảnh theo số người, cứ mỗi nhóm 5 ảnh liền nhau thì xáo ngẫu nhiên (seed cố định) rồi chia cho 5 fold. Nhờ vậy các fold có phân bố số người gần như nhau, val MAE giữa các fold so sánh được với nhau.

In [ ]:
def make_folds(recs, k, seed):
    rng = np.random.RandomState(seed)
    order = np.argsort([len(r['points']) for r in recs], kind='stable')
    fold_of = np.empty(len(recs), int)
    for start in range(0, len(order), k):
        group = order[start:start + k]
        fold_of[group] = rng.permutation(k)[:len(group)]
    return fold_of


FOLD_OF = make_folds(train_recs, K_FOLDS, SEED)


def fold_split(k):
    # (train, val) của fold k
    return ([r for r, f in zip(train_recs, FOLD_OF) if f != k], [r for r, f in zip(train_recs, FOLD_OF) if f == k])


fold_table = pd.DataFrame({f'fold {k}': describe(fold_split(k)[1]) for k in range(K_FOLDS)}).T
fold_table[BIN_LABELS] = [density_bin(pd.Series([len(r['points']) for r in fold_split(k)[1]])).value_counts()
                          .reindex(BIN_LABELS).values for k in range(K_FOLDS)]
fold_table.to_csv(os.path.join(WORK_DIR, f'cv_folds_part{PART}.csv'))
print('Mỗi dòng là tập val của một fold (4 fold còn lại dùng để train):')
display(fold_table)

## 2. Density map ground truth và vùng "có người"

- **Density map:** một Gaussian sigma 15 px tại mỗi đầu người, cắt ở bán kính 3 sigma và chuẩn hóa để tổng bằng 1 cho mỗi người. Khi train, density được cộng gộp xuống 1/4 cho khớp đầu ra của MobileCount.
- **Vùng "có người":** các ô density 1/4 lớn hơn `MASK_THRESH`, tức nằm trong bán kính 3 sigma của ít nhất một đầu người. Phần còn lại là **nền**: ở đó density ground truth bằng đúng 0.

Vùng này dùng để đo **đếm nhầm ở nền** (tổng density dự đoán rơi vào nền, tức chỗ cách mọi đầu người hơn 3 sigma), và làm nhãn cho nhánh mask. Ngưỡng được chọn để phủ trọn Gaussian: nếu ngưỡng cao hơn, phần đuôi Gaussian rơi ra "nền", khiến cả model hoàn hảo cũng bị tính là đếm nhầm.

In [ ]:
def gaussian_kernel(sigma):
    r = int(math.ceil(3 * sigma))
    ax = np.arange(-r, r + 1, dtype=np.float32)
    g = np.exp(-ax ** 2 / (2 * sigma ** 2))
    k = np.outer(g, g)
    return (k / k.sum()).astype(np.float32)


KERNEL = gaussian_kernel(FIXED_SIGMA)


def make_density(points, h, w):
    # Mỗi người là một Gaussian có tổng bằng 1, nên tổng density = số người.
    den = np.zeros((h, w), np.float32)
    r = KERNEL.shape[0] // 2
    for x, y in points:
        x, y = min(max(int(x), 0), w - 1), min(max(int(y), 0), h - 1)
        y0, y1, x0, x1 = max(y - r, 0), min(y + r + 1, h), max(x - r, 0), min(x + r + 1, w)
        patch = KERNEL[y0 - y + r:y1 - y + r, x0 - x + r:x1 - x + r]
        den[y0:y1, x0:x1] += patch / patch.sum()
    return den


def sum_pool(den, s):
    # Thu nhỏ density s lần mỗi chiều bằng cách cộng từng ô s×s (giữ nguyên tổng).
    h, w = den.shape
    H, W = math.ceil(h / s) * s, math.ceil(w / s) * s
    den = np.pad(den, ((0, H - h), (0, W - w)))
    return den.reshape(H // s, s, W // s, s).sum(axis=(1, 3))


t0 = time.time()
densities = {r['name']: make_density(r['points'], r['h'], r['w']) for r in train_recs}
GT_TEST = {r['name']: sum_pool(make_density(r['points'], r['h'], r['w']), STRIDE) for r in test_recs}
print(f'Tạo density cho {len(densities)} ảnh train và {len(GT_TEST)} ảnh test trong {time.time() - t0:.1f}s')
print(f"Sai lệch lớn nhất giữa tổng density và số nhãn: "
      f"{max(abs(densities[r['name']].sum() - len(r['points'])) for r in train_recs):.4f} người")
cover = [(g > MASK_THRESH).mean() for g in GT_TEST.values()]
leak = [g[g <= MASK_THRESH].sum() for g in GT_TEST.values()]
print(f'Vùng "có người" chiếm trung bình {np.mean(cover):.1%} diện tích ảnh test (ít nhất {np.min(cover):.1%}, '
      f'nhiều nhất {np.max(cover):.1%}); density GT nằm ngoài vùng này: {max(leak):.6f} người (phải bằng 0)')

In [ ]:
picks = sorted(train_recs, key=lambda r: len(r['points']))
picks = [picks[0], picks[len(picks) // 2], picks[-1]]
fig, axes = plt.subplots(len(picks), 3, figsize=(16, 4.3 * len(picks)), squeeze=False)
for row, r in zip(axes, picks):
    img = Image.open(r['path']).convert('RGB')
    small = sum_pool(densities[r['name']], STRIDE)
    mask = small > MASK_THRESH
    extent = (0, small.shape[1] * STRIDE, small.shape[0] * STRIDE, 0)
    row[0].imshow(img); row[0].scatter(r['points'][:, 0], r['points'][:, 1], s=3, c='red')
    row[0].set_title(f"{r['name']}: {len(r['points'])} người")
    row[1].imshow(small, cmap='jet'); row[1].set_title(f'Density 1/4 (tổng {small.sum():.1f})')
    row[2].imshow(img); row[2].imshow(mask, cmap='Greens', alpha=0.45, extent=extent)
    row[2].contour(mask, levels=[0.5], colors='lime', linewidths=1.5, extent=extent, origin='upper')
    row[2].set(xlim=(0, img.size[0]), ylim=(img.size[1], 0))
    row[2].set_title(f'Vùng "có người" ({mask.mean():.0%} diện tích); ngoài viền = nền')
    for ax in row:
        ax.axis('off')
plt.suptitle('Ảnh thưa nhất, trung bình và đông nhất trong tập train'); plt.tight_layout(); plt.show()

## 3. Dataset

Mỗi lần lấy một ảnh: cắt ngẫu nhiên một patch, lật ngang với xác suất 50%, chuẩn hóa màu theo ImageNet. Mỗi patch đi kèm density 1/4 và nhãn vùng "có người" (dùng cho nhánh mask). Hai model dùng chung dataset này.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class CrowdPatches(Dataset):
    # Trả về (ảnh, density 1/4 đã nhân DENSITY_SCALE, vùng "có người" 1/4).

    def __init__(self, recs, densities):
        self.recs, self.densities = recs, densities

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        img, den = Image.open(r['path']).convert('RGB'), self.densities[r['name']]
        arr = np.asarray(img, np.float32) / 255.0
        h, w = den.shape
        y, x = random.randint(0, max(h - CROP, 0)), random.randint(0, max(w - CROP, 0))
        arr, den = arr[y:y + CROP, x:x + CROP], den[y:y + CROP, x:x + CROP]
        arr = (arr - MEAN) / STD
        ph, pw = CROP - den.shape[0], CROP - den.shape[1]
        if ph or pw:
            arr = np.pad(arr, ((0, ph), (0, pw), (0, 0)))
            den = np.pad(den, ((0, ph), (0, pw)))
        if random.random() < 0.5:
            arr, den = arr[:, ::-1], den[:, ::-1]
        small = sum_pool(den, STRIDE)
        return (torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1), dtype=np.float32)),
                torch.from_numpy(np.ascontiguousarray(small[None] * DENSITY_SCALE, dtype=np.float32)),
                torch.from_numpy(np.ascontiguousarray((small > MASK_THRESH)[None], dtype=np.float32)))


def make_loader(recs):
    return DataLoader(CrowdPatches(recs, densities), batch_size=BATCH_SIZE, shuffle=True,
                      drop_last=len(recs) >= BATCH_SIZE, num_workers=NUM_WORKERS,
                      pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0)


def denormalize(chw):
    return np.clip(chw.transpose(1, 2, 0) * STD + MEAN, 0, 1)

In [ ]:
seed_everything(SEED)
ds = CrowdPatches(train_recs, densities)
n_show = min(4, len(train_recs))
fig, axes = plt.subplots(2, n_show, figsize=(4.5 * n_show, 8.5), squeeze=False)
for j in range(n_show):
    img, den, _ = ds[j]
    axes[0, j].imshow(denormalize(img.numpy())); axes[0, j].set_title(f'patch {CROP}×{CROP}')
    axes[1, j].imshow(den[0].numpy(), cmap='jet')
    axes[1, j].set_title(f'density {den.shape[-1]}×{den.shape[-2]}: {den.sum() / DENSITY_SCALE:.1f} người')
    for ax in axes[:, j]:
        ax.axis('off')
plt.suptitle('Patch train sau khi cắt ngẫu nhiên và lật ngang'); plt.tight_layout(); plt.show()

## 4. Kiến trúc

```
Ảnh → MobileNetV2 (đặc trưng 1/4, 1/8, 1/16, 1/32)
        1/32 → conv 1×1 → CRP ─┐
        1/16 → conv 1×1 ── + ← phóng to → CRP ─┐
        1/8  → conv 1×1 ── + ← phóng to → CRP ─┐
        1/4  → conv 1×1 ── + ← phóng to → CRP → Y
Y ─┬→ head density → D
   └→ [Mask: head mask → M = sigmoid(·)]        Đầu ra = D × M (baseline: D)
```

- **Baseline MobileCount:** encoder MobileNetV2 pretrained ImageNet, decoder Light-Weight RefineNet (CRP: chuỗi max pool 5×5 + conv 1×1), đầu ra density 1/4 kích thước ảnh. Số người = tổng density.
- **MobileCount + Mask (nhánh mask đám đông):** thêm một head nhỏ (2 lớp conv) dự đoán xác suất mỗi ô 4×4 pixel thuộc vùng có người. Density cuối = D × M, nên density ở nền bị triệt tiêu. Loss = MSE(D × M, density GT) + `MASK_LAMBDA` × BCE(M, nhãn vùng "có người"). Nhãn mask tạo tự động từ density GT, không cần gán nhãn thêm. Khi triển khai, M là bản đồ khoanh vùng đám đông.

In [ ]:
class ChainedResidualPool(nn.Module):
    # Chuỗi max pool 5×5 + conv 1×1 cộng dồn: gom ngữ cảnh rộng với rất ít tham số (Light-Weight RefineNet).

    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class MobileCount(nn.Module):
    # MobileCount, tùy chọn nhánh mask đám đông (mask=True). Tắt mask thì đúng là baseline.

    def __init__(self, pretrained=True, mask=False, channels=64):
        super().__init__()
        weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        f = torchvision.models.mobilenet_v2(weights=weights).features
        self.encoder = nn.ModuleList([f[:4], f[4:7], f[7:14], f[14:18]])   # 1/4, 1/8, 1/16, 1/32
        enc_ch = (24, 32, 96, 320)
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in enc_ch[:3]])
        self.deep = nn.Sequential(nn.Conv2d(enc_ch[3], channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(channels, 1, 1))
        self.mask_head = None
        if mask:
            self.mask_head = nn.Sequential(nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(inplace=True),
                                           nn.Conv2d(32, 1, 1))
            nn.init.constant_(self.mask_head[-1].bias, 1.0)

    def forward(self, x, with_mask=False):
        feats = []
        for stage in self.encoder:
            x = stage(x)
            feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        den = self.head(y)
        if self.mask_head is None:
            return (den, None) if with_mask else den
        logit = self.mask_head(y)
        den = den * torch.sigmoid(logit)
        return (den, logit) if with_mask else den


def build_model(key, pretrained=True):
    return MobileCount(pretrained=pretrained, mask=VARIANTS[key]['mask']).to(device)

In [ ]:
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


@torch.no_grad()
def measure_fps(model, size, half=False, runs=30):
    x = torch.randn(1, 3, *size, device=device)
    ctx = torch.autocast('cuda', dtype=torch.float16) if half and device.type == 'cuda' else contextlib.nullcontext()
    with ctx:
        for _ in range(5):
            model(x)
        sync(); t = time.perf_counter()
        for _ in range(runs):
            model(x)
        sync()
    return runs / (time.perf_counter() - t)


@torch.no_grad()
def count_gmacs(model, size):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as counter:
            model(torch.zeros(1, 3, *size, device=device))
        return counter.get_total_flops() / 2e9
    except Exception as e:
        print('  không đo được FLOPs:', e)
        return None


MODEL_STATS = {}
for key in VARIANTS:
    model = build_model(key).eval()
    with torch.no_grad():
        out = model(torch.zeros(1, 3, CROP, CROP, device=device))
    assert out.shape[-2:] == (CROP // STRIDE, CROP // STRIDE), f'{key}: kích thước đầu ra sai {tuple(out.shape)}'
    MODEL_STATS[key] = {'params_M': sum(p.numel() for p in model.parameters()) / 1e6,
                        'GMACs': count_gmacs(model, FPS_SIZE),
                        'FPS': measure_fps(model, FPS_SIZE), 'FPS_fp16': measure_fps(model, FPS_SIZE, half=True)}
    del model
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 5. Công cụ train và đánh giá

Giống nhau cho cả hai model và mọi fold.

- **Train:** Adam, mixed precision, learning rate giảm theo cosine từ `LR` xuống `LR_MIN` trong `EPOCHS` epoch, trên 320 ảnh của 4 fold.
- **Chọn checkpoint:** mỗi `VAL_EVERY` epoch đo MAE trên 80 ảnh val của fold (ảnh nguyên kích thước); giữ checkpoint có val MAE thấp nhất. Val 80 ảnh ít nhiễu hơn nhiều so với 40 ảnh, và kết quả cuối là trung bình 5 fold nên độ may rủi của một lần chọn được san bớt.
- **Đánh giá test một lần** với checkpoint đã chọn, trên 316 ảnh kích thước gốc:
  - **MAE, MSE** (theo quy ước của lĩnh vực đếm người, MSE thực chất là căn bậc hai sai số bình phương trung bình).
  - **GAME(L):** chia ảnh thành lưới 2^L × 2^L ô, cộng sai số đếm từng ô; đo cả việc đếm đúng chỗ.
  - **Đếm nhầm ở nền:** tổng density dự đoán ở các ô cách mọi đầu người hơn 3 sigma.
  - **IoU mask** (chỉ MobileCount + Mask): độ trùng giữa vùng đám đông dự đoán và thực tế.
- Ngoài checkpoint chọn theo val, notebook cũng ghi lại MAE test của **checkpoint epoch cuối** để đối chiếu (không dùng để chọn gì).

In [ ]:
METRICS = ['MAE', 'MSE', 'GAME1', 'GAME2', 'GAME3', 'Đếm nhầm ở nền', 'IoU mask']


@torch.no_grad()
def predict(model, path, with_mask=False):
    # Density dự đoán (đã chia DENSITY_SCALE) và mask xác suất (nếu có) cho cả ảnh.
    img = (np.asarray(Image.open(path).convert('RGB'), np.float32) / 255.0 - MEAN) / STD
    h, w, _ = img.shape
    H, W = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    img = np.pad(img, ((0, H - h), (0, W - w), (0, 0)))
    x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))[None].to(device)
    hh, ww = math.ceil(h / STRIDE), math.ceil(w / STRIDE)
    if with_mask:
        den, logit = model(x, with_mask=True)
        mask = None if logit is None else torch.sigmoid(logit)[0, 0].float().cpu().numpy()[:hh, :ww]
    else:
        den, mask = model(x), None
    return den[0, 0].float().cpu().numpy()[:hh, :ww] / DENSITY_SCALE, mask


def count_errors(model, recs):
    # Sai số đếm (dự đoán − thật) trên từng ảnh nguyên kích thước.
    model.eval()
    err = np.array([predict(model, r['path'])[0].sum() - len(r['points']) for r in recs])
    model.train()
    return err


def game(pred, gt, level):
    n = 2 ** level
    return sum(abs(p2.sum() - g2.sum())
               for p1, g1 in zip(np.array_split(pred, n, 0), np.array_split(gt, n, 0))
               for p2, g2 in zip(np.array_split(p1, n, 1), np.array_split(g1, n, 1)))


def evaluate_test(model, with_mask=False):
    model.eval()
    rows = []
    for r in test_recs:
        gt = GT_TEST[r['name']]
        pred, mask = predict(model, r['path'], with_mask)
        background = gt <= MASK_THRESH
        row = {'name': r['name'], 'gt': len(r['points']), 'pred': float(pred.sum()),
               'bg_count': float(np.clip(pred, 0, None)[background].sum())}
        for level in (1, 2, 3):
            row[f'GAME{level}'] = float(game(pred, gt, level))
        if mask is not None:
            m, g = mask > 0.5, ~background
            row['mask_iou'] = float((m & g).sum() / max((m | g).sum(), 1))
        rows.append(row)
    df = pd.DataFrame(rows)
    df['error'] = df['pred'] - df['gt']
    return df


def summarize(df):
    out = {'MAE': df['error'].abs().mean(), 'MSE': np.sqrt((df['error'] ** 2).mean()),
           **{f'GAME{l}': df[f'GAME{l}'].mean() for l in (1, 2, 3)}, 'Đếm nhầm ở nền': df['bg_count'].mean()}
    if 'mask_iou' in df:
        out['IoU mask'] = df['mask_iou'].mean()
    return {k: float(v) for k, v in out.items()}


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


SAMPLE_NAMES = [test_recs[i]['name'] for i in np.linspace(0, len(test_recs) - 1, N_SAMPLES).astype(int)]
RESULTS = {}   # RESULTS[(key, fold)]

In [ ]:
def run_tag(key, fold):
    return f'mc_{key}_f{fold}_part{PART}'


def run_files(key, fold):
    tag = run_tag(key, fold)
    return {'results': f'results_{tag}.json', 'preds': f'preds_{tag}.csv', 'history': f'history_{tag}.csv',
            'samples': f'samples_{tag}.npz', 'weights': f'{tag}.pth'}


def load_run(key, fold, folder=WORK_DIR):
    f = {k: os.path.join(folder, v) for k, v in run_files(key, fold).items()}
    with open(f['results'], encoding='utf-8') as fh:
        res = json.load(fh)
    with np.load(f['samples']) as s:
        samples = {k: s[k] for k in s.files}
    return {'res': res, 'preds': pd.read_csv(f['preds']), 'history': pd.read_csv(f['history']), 'samples': samples}


def find_previous(key, fold):
    # Kết quả fold này đã có (ở WORK_DIR hoặc output của version trước được Add Input)?
    name = run_files(key, fold)['results']
    for path in [os.path.join(WORK_DIR, name)] + glob.glob(os.path.join(INPUT_DIR, '**', name), recursive=True):
        if os.path.exists(path):
            with open(path, encoding='utf-8') as fh:
                res = json.load(fh)
            if not res.get('quick_test') and res.get('config', {}).get('epochs') == EPOCHS:
                return os.path.dirname(path)
    return None


def train_model(key, fold):
    cfg, files = VARIANTS[key], {k: os.path.join(WORK_DIR, v) for k, v in run_files(key, fold).items()}
    prev = find_previous(key, fold)
    if prev:
        if os.path.abspath(prev) != os.path.abspath(WORK_DIR):
            for fname in run_files(key, fold).values():
                if os.path.exists(os.path.join(prev, fname)):
                    shutil.copy(os.path.join(prev, fname), WORK_DIR)
        rec = load_run(key, fold)
        print(f"{cfg['label']}, fold {fold}: dùng lại kết quả đã có (val MAE {rec['res']['val_MAE']:.2f}, "
              f"test MAE {rec['res']['MAE']:.2f}).")
        return rec

    tr, va = fold_split(fold)
    seed = SEED + fold
    print(f"{cfg['label']}, fold {fold}: train {len(tr)} ảnh, val {len(va)} ảnh, {EPOCHS} epoch, seed {seed}. "
          f"Loss: density" + (' + mask' if cfg['mask'] else ''))
    seed_everything(seed)
    model = build_model(key)
    loader = make_loader(tr)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    use_amp = device.type == 'cuda'
    scaler = make_scaler(use_amp)

    history, best_mae, best_epoch, start = [], float('inf'), 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_epoch, parts = time.time(), {'loss': [], 'loss_density': [], 'loss_mask': []}
        for img, den, mask_gt in loader:
            img, den, mask_gt = (t.to(device, non_blocking=True) for t in (img, den, mask_gt))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out, logit = model(img, with_mask=True)
            loss = F.mse_loss(out.float(), den)
            parts['loss_density'].append(loss.item())
            if logit is not None:
                l_mask = F.binary_cross_entropy_with_logits(logit.float(), mask_gt)
                loss = loss + MASK_LAMBDA * l_mask
                parts['loss_mask'].append(l_mask.item())
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            parts['loss'].append(loss.item())
        row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], 'epoch_s': time.time() - t_epoch,
               **{k: float(np.mean(v)) if v else np.nan for k, v in parts.items()}}
        scheduler.step()
        time_over = (time.time() - start) / 3600 > MAX_HOURS_PER_RUN
        if epoch % VAL_EVERY == 0 or epoch == EPOCHS or time_over:
            err = count_errors(model, va)
            row['val_MAE'], row['val_MSE'] = float(np.abs(err).mean()), float(np.sqrt((err ** 2).mean()))
            if row['val_MAE'] < best_mae:
                best_mae, best_epoch = row['val_MAE'], epoch
                torch.save(model.state_dict(), files['weights'])
        if epoch % LOG_EVERY == 0 or epoch == EPOCHS or time_over:
            print(f"  epoch {epoch:3d} | lr {row['lr']:.1e} | loss {row['loss']:.5f} | "
                  f"val MAE {row.get('val_MAE', np.nan):6.2f} (tốt nhất {best_mae:.2f} ở epoch {best_epoch}) | "
                  f"{row['epoch_s']:4.1f}s/epoch | {(time.time() - start) / 60:5.1f} phút")
        history.append(row)
        if time_over:
            print(f'  CẢNH BÁO: dừng ở epoch {epoch}/{EPOCHS} vì chạm giới hạn {MAX_HOURS_PER_RUN} giờ.')
            break
    train_minutes = (time.time() - start) / 60
    history = pd.DataFrame(history)
    history.to_csv(files['history'], index=False)

    # Đối chiếu: test của checkpoint epoch cuối (không dùng để chọn gì)
    last_test_mae = float(np.abs(count_errors(model, test_recs)).mean())

    # Đánh giá test một lần với checkpoint chọn theo val
    model.load_state_dict(torch.load(files['weights'], map_location=device))
    test_df = evaluate_test(model, with_mask=cfg['mask'])
    test_df.to_csv(files['preds'], index=False)
    samples = {}
    for i, n in enumerate(SAMPLE_NAMES):
        den, mask = predict(model, test_by_name[n]['path'], with_mask=cfg['mask'])
        samples[f'den_{i}'] = den.astype(np.float32)
        if mask is not None:
            samples[f'mask_{i}'] = mask.astype(np.float32)
    np.savez_compressed(files['samples'], **samples)

    res = {'variant': key, 'label': cfg['label'], 'fold': fold, 'seed': seed, 'part': PART,
           'val_MAE': best_mae, 'best_epoch': best_epoch, **summarize(test_df), 'MAE_last_epoch': last_test_mae,
           'device': GPU_NAME, 'epochs_run': int(history['epoch'].max()), 'train_minutes': train_minutes,
           'quick_test': QUICK,
           'config': {**cfg, 'epochs': EPOCHS, 'lr': LR, 'lr_min': LR_MIN, 'batch_size': BATCH_SIZE, 'crop': CROP,
                      'mask_lambda': MASK_LAMBDA, 'k_folds': K_FOLDS, 'n_train': len(tr), 'n_val': len(va),
                      'checkpoint': 'val MAE thấp nhất của fold'}}
    with open(files['results'], 'w', encoding='utf-8') as fh:
        json.dump(res, fh, ensure_ascii=False, indent=2)
    print(f"  → xong sau {train_minutes:.1f} phút: epoch tốt nhất {best_epoch} (val MAE {best_mae:.2f}) → "
          f"test MAE {res['MAE']:.2f}, MSE {res['MSE']:.2f} | epoch cuối: test MAE {last_test_mae:.2f}\n")
    del model, optimizer, loader
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return load_run(key, fold)

## 6. Train 5 fold × 2 model

Mỗi fold train baseline rồi MobileCount + Mask (cùng fold, cùng seed). Fold đã có kết quả (từ phiên trước) được dùng lại. Trước mỗi lần train, notebook ước lượng thời gian còn lại của phiên; nếu có thể vượt `SESSION_HOURS` thì dừng để Kaggle kịp lưu output, phần còn lại chạy ở version sau.

In [ ]:
run_minutes, stopped = [], False   # thời gian các lần train thật trong phiên này
for fold in RUN_FOLDS:
    for key in VARIANTS:
        is_new = find_previous(key, fold) is None
        elapsed = (time.time() - SESSION_START) / 3600
        need = (max(run_minutes) / 60 * 1.1) if run_minutes else 0.8
        if is_new and elapsed + need > SESSION_HOURS:
            print(f'Dừng: đã chạy {elapsed:.1f} giờ, lần train tiếp theo (~{need:.1f} giờ) có thể vượt {SESSION_HOURS} giờ. '
                  f'Tạo version mới, Add Input output này để chạy tiếp.')
            stopped = True
            break
        RESULTS[(key, fold)] = train_model(key, fold)
        if is_new:
            run_minutes.append(RESULTS[(key, fold)]['res']['train_minutes'])
    if stopped:
        break

# Nạp thêm các fold đã có từ phiên trước nhưng không nằm trong RUN_FOLDS
for fold in range(K_FOLDS):
    for key in VARIANTS:
        if (key, fold) not in RESULTS and find_previous(key, fold):
            RESULTS[(key, fold)] = train_model(key, fold)

done = sorted({f for (_, f) in RESULTS})
full = [f for f in done if all((k, f) in RESULTS for k in VARIANTS)]
print(f'Fold đã đủ hai model: {full} / {K_FOLDS}')

## 7. Hình kết quả

Hình được lưu vào `/kaggle/working` (`cv_*.png`) để đưa vào báo cáo. Chỉ dùng các fold đã có đủ hai model.

**Hình 1. Quá trình train:** loss density và val MAE theo epoch, trung bình 5 fold (vùng mờ: min–max qua các fold), và learning rate.

**Hình 2. Kết quả từng fold:** test MAE và đếm nhầm ở nền của hai model trên từng fold. Hai điểm cùng fold được nối bằng đường thẳng để thấy Mask tốt hơn hay kém hơn ở từng fold.

**Hình 3. Sai số theo nhóm mật độ:** MAE và đếm nhầm ở nền theo số người thật, trung bình 5 fold (thanh lỗi: độ lệch chuẩn qua fold).

**Hình 4. Density map và khoanh vùng** trên các ảnh test cố định, dùng model của **fold 0** (chọn cố định trước, không chọn theo kết quả test).

In [ ]:
def save_fig(name):
    plt.savefig(os.path.join(WORK_DIR, f'cv_{name}_part{PART}.png'), bbox_inches='tight')


keys = list(VARIANTS)
folds = full
assert folds, 'Chưa có fold nào đủ hai model.'

# Hình 1: quá trình train
fig, axes = plt.subplots(1, 3, figsize=(22, 4.5))
for k in keys:
    l = VARIANTS[k]['label']
    loss = pd.concat([RESULTS[(k, f)]['history'].set_index('epoch')['loss_density'] for f in folds], axis=1)
    val = pd.concat([RESULTS[(k, f)]['history'].dropna(subset=['val_MAE']).set_index('epoch')['val_MAE']
                     for f in folds], axis=1)
    axes[0].plot(loss.index, loss.mean(axis=1), label=l)
    line, = axes[1].plot(val.index, val.mean(axis=1), label=l)
    axes[1].fill_between(val.index, val.min(axis=1), val.max(axis=1), alpha=0.15, color=line.get_color())
axes[0].set(title='Loss density khi train (TB các fold)', xlabel='epoch', yscale='log'); axes[0].legend(fontsize=8)
axes[1].set(title='Val MAE (TB các fold, vùng mờ: min–max)', xlabel='epoch', ylabel='MAE', yscale='log')
axes[1].legend(fontsize=8)
h0 = RESULTS[(keys[0], folds[0])]['history']
axes[2].plot(h0['epoch'], h0['lr']); axes[2].set(title='Learning rate (cosine)', xlabel='epoch', yscale='log')
plt.tight_layout(); save_fig('curves'); plt.show()

# Hình 2: kết quả từng fold
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, metric, title in ((axes[0], 'MAE', 'Test MAE theo fold'), (axes[1], 'Đếm nhầm ở nền', 'Đếm nhầm ở nền theo fold')):
    for f in folds:
        ax.plot([0, 1], [RESULTS[(k, f)]['res'][metric] for k in keys], 'o-', color='gray', alpha=0.6)
        ax.annotate(f'fold {f}', (1.03, RESULTS[(keys[1], f)]['res'][metric]), fontsize=8, va='center')
    ax.set_xticks([0, 1], [VARIANTS[k]['label'] for k in keys]); ax.set_xlim(-0.3, 1.4)
    ax.set(title=title, ylabel=metric)
plt.tight_layout(); save_fig('folds'); plt.show()

# Hình 3: sai số theo nhóm mật độ (trung bình qua fold)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.5))
for ax, col, agg, title, ylabel in (
        (axes[0], 'error', lambda e: e.abs().mean(), 'MAE theo nhóm mật độ', 'MAE'),
        (axes[1], 'bg_count', 'mean', 'Đếm nhầm ở nền theo nhóm mật độ', 'người / ảnh')):
    mean, std = {}, {}
    for k in keys:
        per = pd.concat([RESULTS[(k, f)]['preds'].pipe(
            lambda d: d.groupby(density_bin(d['gt']), observed=False)[col].agg(agg)) for f in folds], axis=1)
        mean[VARIANTS[k]['label']], std[VARIANTS[k]['label']] = per.mean(axis=1), per.std(axis=1)
    pd.DataFrame(mean).plot.bar(ax=ax, rot=0, yerr=pd.DataFrame(std), capsize=3)
    ax.set(title=title + ' (TB 5 fold)', xlabel='số người thật', ylabel=ylabel)
plt.tight_layout(); save_fig('density_bins'); plt.show()

# Hình 4: density map và bản đồ khoanh vùng (fold 0)
f0 = 0 if 0 in folds else folds[0]
mask_key = next(k for k in keys if VARIANTS[k]['mask'])
n_cols = 2 + len(keys) + 1
fig, axes = plt.subplots(len(SAMPLE_NAMES), n_cols, figsize=(4.4 * n_cols, 3.6 * len(SAMPLE_NAMES)), squeeze=False)
for row, (i, name) in zip(axes, enumerate(SAMPLE_NAMES)):
    r = test_by_name[name]
    img = Image.open(r['path']).convert('RGB')
    w, h = img.size
    gt = GT_TEST[name]
    extent = (0, gt.shape[1] * STRIDE, gt.shape[0] * STRIDE, 0)
    row[0].imshow(img); row[0].set_title(f"{name}: {len(r['points'])} người", fontsize=10)
    row[1].imshow(img); row[1].imshow(gt, cmap='jet', alpha=0.5, extent=extent); row[1].set_title('Ground truth', fontsize=10)
    for ax, k in zip(row[2:], keys):
        den = RESULTS[(k, f0)]['samples'][f'den_{i}']
        ax.imshow(img); ax.imshow(den, cmap='jet', alpha=0.5, extent=extent)
        ax.set_title(f"{VARIANTS[k]['label']}: {den.sum():.1f} ({den.sum() - len(r['points']):+.1f})", fontsize=9)
    ax = row[-1]
    mask = RESULTS[(mask_key, f0)]['samples'][f'mask_{i}']
    ax.imshow(img)
    ax.imshow(mask, cmap='Greens', alpha=0.45, vmin=0, vmax=1, extent=extent)
    ax.contour(mask, levels=[0.5], colors='lime', linewidths=1.5, extent=extent, origin='upper')
    ax.contour(gt > MASK_THRESH, levels=[0.5], colors='red', linewidths=1, linestyles='--', extent=extent, origin='upper')
    ax.set_title('Khoanh vùng (Mask): dự đoán (xanh), thật (đỏ)', fontsize=9)
    for ax in row:
        ax.set(xlim=(0, w), ylim=(h, 0)); ax.axis('off')
plt.tight_layout(); save_fig('visual'); plt.show()

## 8. Bảng so sánh

**Bảng chính:** trung bình ± độ lệch chuẩn qua các fold; mỗi fold là kết quả trên toàn bộ 316 ảnh test của checkpoint chọn theo val MAE của fold đó.

| Cột | Ý nghĩa | Tốt hơn khi |
|---|---|---|
| Val MAE | MAE trên 80 ảnh val của fold (dùng để chọn checkpoint) | thấp |
| MAE, MSE | sai số đếm trên tập test | thấp |
| GAME1–3 | chia ảnh thành 4, 16, 64 ô và cộng sai số từng ô: đo cả việc đếm đúng chỗ | thấp |
| Đếm nhầm ở nền | số người model đặt vào chỗ cách mọi đầu người hơn 3 sigma | thấp |
| IoU mask | độ trùng giữa vùng đám đông dự đoán và thực tế (chỉ MobileCount + Mask) | cao |
| MAE epoch cuối | test MAE của checkpoint epoch 400, chỉ để đối chiếu | thấp |
| Tham số, GMACs, FPS | kích thước, khối lượng tính toán và tốc độ (ảnh 1024×768, batch 1) | thấp / cao |

**So sánh từng cặp:** ở mỗi fold, lấy chỉ số của Mask trừ baseline. Bảng cho trung bình ± độ lệch chuẩn của hiệu và số fold mà Mask tốt hơn.

**Trọng số cho demo:** với mỗi model, chọn fold có **val** MAE thấp nhất (không nhìn test) và lưu thành `mc_{model}_partB.pth`.

In [ ]:
COLS = ['val_MAE', 'MAE', 'MSE', 'GAME1', 'GAME2', 'GAME3', 'Đếm nhầm ở nền', 'IoU mask', 'MAE_last_epoch']
NAMES = {'val_MAE': 'Val MAE', 'MAE_last_epoch': 'MAE epoch cuối'}

# Từng fold
per_fold = pd.DataFrame([{'Model': VARIANTS[k]['label'], 'Fold': f, 'Epoch tốt nhất': RESULTS[(k, f)]['res']['best_epoch'],
                          **{NAMES.get(c, c): RESULTS[(k, f)]['res'].get(c, np.nan) for c in COLS}}
                         for k in keys for f in folds]).set_index(['Model', 'Fold'])
per_fold.to_csv(os.path.join(WORK_DIR, f'cv_per_fold_part{PART}.csv'))
print('Kết quả từng fold:')
display(per_fold.round(2))

# Bảng chính: trung bình ± độ lệch chuẩn
summary = {}
for k in keys:
    d = per_fold.loc[VARIANTS[k]['label']]
    st = MODEL_STATS[k]
    summary[VARIANTS[k]['label']] = {
        **{c: (f'{d[c].mean():.2f} ± {d[c].std():.2f}' if d[c].notna().any() else '–') for c in d.columns if c != 'Epoch tốt nhất'},
        'Tham số (triệu)': f"{st['params_M']:.2f}", 'GMACs': f"{st['GMACs']:.2f}" if st['GMACs'] else '–',
        'FPS fp32 / fp16': f"{st['FPS']:.1f} / {st['FPS_fp16']:.1f}"}
summary = pd.DataFrame(summary).T
summary.to_csv(os.path.join(WORK_DIR, f'cv_summary_part{PART}.csv'))
print(f'\nPart {PART}, {len(folds)} fold, 316 ảnh test, checkpoint theo val MAE của fold, {GPU_NAME}.'
      + (' CHẠY THỬ, KHÔNG DÙNG SỐ LIỆU.' if QUICK else ''))
display(summary)

# So sánh từng cặp (Mask − baseline)
base, mask = (per_fold.loc[VARIANTS[k]['label']] for k in keys)
pair = {}
for c, better in (('MAE', 'thấp'), ('MSE', 'thấp'), ('GAME3', 'thấp'), ('Đếm nhầm ở nền', 'thấp'), ('Val MAE', 'thấp')):
    diff = mask[c] - base[c]
    pair[c] = {'Mask − baseline (TB ± ĐLC)': f'{diff.mean():+.2f} ± {diff.std():.2f}',
               'Tương đối': f'{diff.mean() / base[c].mean():+.1%}',
               'Số fold Mask tốt hơn': f'{int((diff < 0).sum())} / {len(diff)}'}
pair = pd.DataFrame(pair).T
pair.to_csv(os.path.join(WORK_DIR, f'cv_paired_part{PART}.csv'))
print('\nSo sánh từng cặp trên cùng fold:')
display(pair)

# Trọng số cho demo: fold có val MAE thấp nhất
for k in keys:
    best_fold = min(folds, key=lambda f: RESULTS[(k, f)]['res']['val_MAE'])
    src = os.path.join(WORK_DIR, run_files(k, best_fold)['weights'])
    if os.path.exists(src):
        shutil.copy(src, os.path.join(WORK_DIR, f'mc_{k}_part{PART}.pth'))
        print(f"{VARIANTS[k]['label']}: dùng fold {best_fold} (val MAE {RESULTS[(k, best_fold)]['res']['val_MAE']:.2f}) "
              f"→ mc_{k}_part{PART}.pth")
    else:
        print(f"{VARIANTS[k]['label']}: không thấy trọng số fold {best_fold} (Add Input cả file .pth của phiên trước).")